[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/083.%20The%20Multi-Facility%20Location%20-%20p-Median%20Problem/P83-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 83. The Multi-Facility Location: p-Median Problem

## Tier 3: The Advanced Algorithm (Tabu Search Implementation)

### Key assumptions
- Tabu list prevents cycling to previously visited solutions
- Aspiration criteria allow tabu moves if they achieve new best solutions
- Adaptive tabu tenure balances exploration and exploitation
- Diversification mechanism enables global search

### Approach (step-by-step)
Tabu Search enhances local search with memory structures:
1. **Initialization**: Generate initial solution and tabu list
2. **Neighborhood Evaluation**: Evaluate all facility swaps
3. **Tabu Management**: Maintain recently performed moves
4. **Aspiration Check**: Allow tabu moves that improve best solution
5. **Adaptive Tenure**: Dynamically adjust memory length
6. **Diversification**: Periodic restart for global exploration
7. **Intensification**: Focus search around promising regions

### What to look for in the results
- Tabu search navigation through solution space
- Adaptive tenure management effects
- Diversification triggers and restart behavior
- Best solution discovery over time
- Comparison with simple local search performance

### Concrete example (from the source)
15-customer, 25-facility problem with 4 facilities to locate, demonstrating tabu search with adaptive memory and diversification strategies.

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Set
import random
import time
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

print("Libraries imported successfully for p-Median Tabu Search")

Libraries imported successfully for p-Median Tabu Search


In [ ]:
class PMedianTabuSearch:
    """
    Tabu Search algorithm for p-Median problem
    Uses adaptive memory and diversification strategies
    """
    
    def __init__(self, customer_positions: List[float], demands: List[float], 
                 facility_positions: List[float], p: int):
        """
        Initialize p-Median Tabu Search solver
        
        Args:
            customer_positions: Geographic positions of customers
            demands: Demand volumes for each customer
            facility_positions: Potential facility locations
            p: Number of facilities to locate
        """
        self.customer_positions = np.array(customer_positions)
        self.demands = np.array(demands)
        self.facility_positions = np.array(facility_positions)
        self.p = p
        self.n_customers = len(customer_positions)
        self.n_facilities = len(facility_positions)
        
        # Calculate distance matrix
        self.distances = self._calculate_distances()
        
        # Tabu search parameters
        self.base_tenure = 7  # Base tabu list length
        self.max_tenure = 15
        self.min_tenure = 3
        self.diversification_trigger = 30  # Iterations without improvement
        self.max_iterations = 200
        
        # Solution tracking
        self.current_solution = None
        self.current_cost = np.inf
        self.best_solution = None
        self.best_cost = np.inf
        
        # Tabu structures
        self.tabu_list = {}  # move -> iteration when it becomes allowed
        self.current_iteration = 0
        
        # Search history
        self.iteration_history = []
        self.best_found_iterations = []
        self.diversification_points = []
        
    def _calculate_distances(self) -> np.ndarray:
        """Calculate Euclidean distance matrix between customers and facilities"""
        distances = np.zeros((self.n_customers, self.n_facilities))
        for i in range(self.n_customers):
            for j in range(self.n_facilities):
                distances[i, j] = abs(self.customer_positions[i] - self.facility_positions[j])
        return distances
    
    def _assign_customers(self, facility_set: Set[int]) -> Dict[int, List[int]]:
        """Assign each customer to nearest open facility"""
        assignments = {fac_idx: [] for fac_idx in facility_set}
        
        for cust_idx in range(self.n_customers):
            min_dist = np.inf
            nearest_facility = -1
            
            for fac_idx in facility_set:
                dist = self.distances[cust_idx, fac_idx]
                if dist < min_dist:
                    min_dist = dist
                    nearest_facility = fac_idx
            
            assignments[nearest_facility].append(cust_idx)
        
        return assignments
    
    def _calculate_total_cost(self, facility_set: Set[int]) -> float:
        """Calculate total transportation cost for given facility set"""
        assignments = self._assign_customers(facility_set)
        total_cost = 0.0
        
        for fac_idx, customers in assignments.items():
            for cust_idx in customers:
                total_cost += self.demands[cust_idx] * self.distances[cust_idx, fac_idx]
        
        return total_cost
    
    def _generate_initial_solution(self) -> Set[int]:
        """Generate initial facility locations using random selection"""
        return set(random.sample(range(self.n_facilities), self.p))
    
    def _calculate_adaptive_tenure(self) -> int:
        """
        Calculate adaptive tabu tenure based on search progress
        
        Returns:
            Adaptive tenure value
        """
        if len(self.iteration_history) < 10:
            return self.base_tenure
        
        # Count recent improvements
        recent_improvements = 0
        for i in range(max(0, len(self.iteration_history) - 10), len(self.iteration_history)):
            if self.iteration_history[i]['improvement'] > 0.001:
                recent_improvements += 1
        
        # Adjust tenure based on improvement frequency
        if recent_improvements >= 3:  # Many improvements, reduce tenure for exploration
            tenure = max(self.min_tenure, self.base_tenure - 2)
        elif recent_improvements <= 1:  # Few improvements, increase tenure for intensification
            tenure = min(self.max_tenure, self.base_tenure + 2)
        else:
            tenure = self.base_tenure
        
        return tenure
    
    def _is_move_tabu(self, facility_out: int, facility_in: int) -> bool:
        """
        Check if a move is currently tabu
        
        Args:
            facility_out: Facility to remove
            facility_in: Facility to add
            
        Returns:
            True if move is tabu
        """
        move = (facility_out, facility_in)
        if move in self.tabu_list:
            return self.current_iteration < self.tabu_list[move]
        return False
    
    def _add_tabu_move(self, facility_out: int, facility_in: int, tenure: int):
        """
        Add a move to the tabu list
        
        Args:
            facility_out: Facility to remove
            facility_in: Facility to add
            tenure: Number of iterations the move remains tabu
        """
        move = (facility_out, facility_in)
        self.tabu_list[move] = self.current_iteration + tenure
    
    def _update_tabu_list(self):
        """Remove expired moves from tabu list"""
        expired_moves = []
        for move, expire_iteration in self.tabu_list.items():
            if self.current_iteration >= expire_iteration:
                expired_moves.append(move)
        
        for move in expired_moves:
            del self.tabu_list[move]
    
    def _find_best_move(self, current_facilities: Set[int]) -> Tuple[int, int, float]:
        """
        Find best move considering tabu status and aspiration criteria
        
        Args:
            current_facilities: Current set of open facilities
            
        Returns:
            Tuple of (facility_out, facility_in, improvement)
        """
        best_move = (-1, -1)
        best_improvement = -np.inf
        current_cost = self._calculate_total_cost(current_facilities)
        
        # Evaluate all possible swaps
        for facility_out in current_facilities:
            for facility_in in range(self.n_facilities):
                if facility_in in current_facilities:
                    continue
                
                # Calculate new cost
                new_facilities = current_facilities.copy()
                new_facilities.remove(facility_out)
                new_facilities.add(facility_in)
                new_cost = self._calculate_total_cost(new_facilities)
                improvement = current_cost - new_cost
                
                # Check if move is tabu
                is_tabu = self._is_move_tabu(facility_out, facility_in)
                
                # Aspiration criteria: allow tabu move if it improves best solution
                if is_tabu and new_cost >= self.best_cost:
                    continue  # Skip tabu move that doesn't improve best solution
                
                # Update best move
                if improvement > best_improvement:
                    best_improvement = improvement
                    best_move = (facility_out, facility_in)
        
        return best_move[0], best_move[1], best_improvement
    
    def _diversify_search(self) -> Set[int]:
        """
        Generate diversified solution for restart
        
        Returns:
            New diversified facility set
        """
        # Create new solution with some randomness
        new_solution = set()
        
        # Keep some facilities from best solution
        if self.best_solution:
            keep_count = max(1, self.p // 2)
            kept_facilities = random.sample(list(self.best_solution), keep_count)
            new_solution.update(kept_facilities)
        
        # Add random facilities to complete the solution
        remaining_facilities = [f for f in range(self.n_facilities) if f not in new_solution]
        additional_facilities = random.sample(remaining_facilities, self.p - len(new_solution))
        new_solution.update(additional_facilities)
        
        return new_solution
    
    def solve(self, max_iterations: int = None, verbose: bool = True) -> Tuple[float, Set[int], Dict[int, List[int]]]:
        """
        Solve the p-Median problem using tabu search
        
        Args:
            max_iterations: Maximum number of iterations
            verbose: Whether to print progress
            
        Returns:
            Tuple of (total_cost, facility_locations, customer_assignments)
        """
        if max_iterations:
            self.max_iterations = max_iterations
        
        if verbose:
            print(f"Solving p-Median with Tabu Search")
            print(f"Customers: {self.n_customers}, Facilities: {self.n_facilities}, p: {self.p}")
            print(f"Base tabu tenure: {self.base_tenure}, Max iterations: {self.max_iterations}")
        
        # Initialize
        self.current_solution = self._generate_initial_solution()
        self.current_cost = self._calculate_total_cost(self.current_solution)
        self.best_solution = self.current_solution.copy()
        self.best_cost = self.current_cost
        self.current_iteration = 0
        
        if verbose:
            print(f"\nInitial solution: {sorted(self.current_solution)}")
            print(f"Initial cost: {self.current_cost:.2f}")
        
        # Store initial state
        self.iteration_history.append({
            'iteration': 0,
            'solution': self.current_solution.copy(),
            'cost': self.current_cost,
            'improvement': 0,
            'best_cost': self.best_cost,
            'tabu_list_size': 0,
            'tenure': self.base_tenure
        })
        
        # Main tabu search loop
        iterations_without_improvement = 0
        improvements_found = 0
        
        while self.current_iteration < self.max_iterations:
            self.current_iteration += 1
            
            # Calculate adaptive tenure
            current_tenure = self._calculate_adaptive_tenure()
            
            # Find best move
            facility_out, facility_in, improvement = self._find_best_move(self.current_solution)
            
            if facility_out >= 0:  # Valid move found
                # Apply move
                self.current_solution.remove(facility_out)
                self.current_solution.add(facility_in)
                self.current_cost -= improvement
                
                # Add move to tabu list
                self._add_tabu_move(facility_out, facility_in, current_tenure)
                
                # Update best solution if improved
                if self.current_cost < self.best_cost:
                    self.best_solution = self.current_solution.copy()
                    self.best_cost = self.current_cost
                    self.best_found_iterations.append(self.current_iteration)
                    improvements_found += 1
                    iterations_without_improvement = 0
                    
                    if verbose and improvements_found <= 5:  # Show first few improvements
                        print(f"Iteration {self.current_iteration}: New best! Swap {facility_out} -> {facility_in}, Cost: {self.current_cost:.2f}")
                else:
                    iterations_without_improvement += 1
                
                # Check for diversification trigger
                if iterations_without_improvement >= self.diversification_trigger:
                    if verbose:
                        print(f"Iteration {self.current_iteration}: Diversification triggered, Cost: {self.current_cost:.2f}")
                    
                    self.diversification_points.append(self.current_iteration)
                    self.current_solution = self._diversify_search()
                    self.current_cost = self._calculate_total_cost(self.current_solution)
                    iterations_without_improvement = 0
                    
                    # Clear tabu list for fresh start
                    self.tabu_list.clear()
            else:
                # No valid move found (shouldn't happen in p-median)
                break
            
            # Update tabu list
            self._update_tabu_list()
            
            # Store iteration data
            self.iteration_history.append({
                'iteration': self.current_iteration,
                'solution': self.current_solution.copy(),
                'cost': self.current_cost,
                'improvement': improvement,
                'best_cost': self.best_cost,
                'tabu_list_size': len(self.tabu_list),
                'tenure': current_tenure
            })
        
        # Get final assignments
        final_assignments = self._assign_customers(self.best_solution)
        
        if verbose:
            print(f"\nFinal best solution: {sorted(self.best_solution)}")
            print(f"Final best cost: {self.best_cost:.2f}")
            print(f"Total improvement: {((self.iteration_history[0]['cost'] - self.best_cost) / self.iteration_history[0]['cost'] * 100):.2f}%")
            print(f"Number of improvements found: {improvements_found}")
            print(f"Diversification events: {len(self.diversification_points)}")
        
        return self.best_cost, self.best_solution, final_assignments
    
    def print_solution(self, total_cost: float, facility_locations: Set[int], 
                      customer_assignments: Dict[int, List[int]]):
        """Print the solution in a readable format"""
        print(f"\n{'='*60}")
        print("TABU SEARCH SOLUTION")
        print(f"{'='*60}")
        print(f"Total Transportation Cost: {total_cost:.2f}")
        print(f"Number of Facilities: {self.p}")
        print(f"\nFacility Locations and Assignments:")
        
        for i, fac_idx in enumerate(sorted(facility_locations)):
            fac_pos = self.facility_positions[fac_idx]
            customers = customer_assignments[fac_idx]
            customer_positions = [self.customer_positions[c] for c in customers]
            customer_demands = [self.demands[c] for c in customers]
            
            print(f"\n  Facility {i+1} at position {fac_pos:.1f} (Index {fac_idx}):")
            print(f"    Serves customers at positions: {customer_positions}")
            print(f"    Customer demands: {customer_demands}")
            
            # Calculate cost for this facility
            cost = sum(self.demands[c] * self.distances[c, fac_idx] for c in customers)
            print(f"    Cost contribution: {cost:.2f}")
            print(f"    Number of customers: {len(customers)}")

print("PMedianTabuSearch class defined successfully")

PMedianTabuSearch class defined successfully


In [ ]:
# Create the concrete example from the source
# 15 customers, 25 facilities, 4 facilities to locate

# Generate test data
n_customers = 15
n_facilities = 25
p = 4

# Customer positions (geographic distribution)
customer_positions = np.random.uniform(0, 30, n_customers)
customer_positions.sort()

# Customer demands (varying volumes)
demands = np.random.uniform(8, 30, n_customers)

# Facility positions (more options than customers)
facility_positions = np.random.uniform(0, 30, n_facilities)
facility_positions.sort()

print("CONCRETE EXAMPLE SETUP")
print(f"Number of customers: {n_customers}")
print(f"Number of potential facilities: {n_facilities}")
print(f"Number of facilities to locate: {p}")
print(f"\nCustomer positions: {customer_positions.round(2).tolist()}")
print(f"Customer demands: {demands.round(2).tolist()}")
print(f"Facility positions: {facility_positions.round(2).tolist()}")

# Create and solve the problem
solver = PMedianTabuSearch(customer_positions.tolist(), demands.tolist(), 
                          facility_positions.tolist(), p)

total_cost, facility_locations, customer_assignments = solver.solve(
    max_iterations=150, verbose=True
)

# Print detailed solution
solver.print_solution(total_cost, facility_locations, customer_assignments)

CONCRETE EXAMPLE SETUP
Number of customers: 15
Number of potential facilities: 25
Number of facilities to locate: 4

Customer positions: [6.32, 8.3, 12.65, 13.16, 14.6, 17.36, 17.77, 18.69, 18.95, 21.76, 21.99, 26.01, 26.2, 27.02, 27.37]
Customer demands: [13.44, 27.16, 17.8, 19.33, 15.9, 21.04, 11.6, 16.6, 29.33, 13.68, 22.45, 15.15, 25.02, 10.88, 29.34]
Facility positions: [1.1, 1.18, 1.25, 1.71, 3.36, 3.66, 5.01, 5.03, 6.18, 8.09, 9.48, 13.57, 14.24, 15.22, 15.94, 17.56, 18.51, 19.91, 21.2, 22.09, 24.17, 25.33, 26.05, 28.02, 28.1]
Solving p-Median with Tabu Search
Customers: 15, Facilities: 25, p: 4
Base tabu tenure: 7, Max iterations: 150

Initial solution: [0, 3, 20, 23]
Initial cost: 1447.23
Iteration 1: New best! Swap 0 -> 15, Cost: 711.39
Iteration 2: New best! Swap 3 -> 9, Cost: 493.81
Iteration 3: New best! Swap 20 -> 11, Cost: 398.66
Iteration 4: New best! Swap 23 -> 22, Cost: 338.47
Iteration 5: New best! Swap 15 -> 16, Cost: 294.96
Iteration 35: Diversification triggered, 

In [ ]:
# Visualize tabu search process
def visualize_tabu_search_process(solver):
    """Visualize the tabu search iteration process"""
    
    if not solver.iteration_history:
        print("No iteration history available")
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # Extract iteration data
    iterations = [h['iteration'] for h in solver.iteration_history]
    costs = [h['cost'] for h in solver.iteration_history]
    best_costs = [h['best_cost'] for h in solver.iteration_history]
    tabu_sizes = [h['tabu_list_size'] for h in solver.iteration_history]
    tenures = [h['tenure'] for h in solver.iteration_history]
    
    # Plot 1: Cost evolution with best solution tracking
    ax1.plot(iterations, costs, 'b-', alpha=0.7, linewidth=1, label='Current Cost')
    ax1.plot(iterations, best_costs, 'r-', linewidth=2, label='Best Cost')
    
    # Mark best solution improvements
    for iter_num in solver.best_found_iterations:
        if iter_num < len(iterations):
            ax1.scatter(iterations[iter_num], best_costs[iter_num], 
                       color='red', s=100, marker='*', zorder=5)
    
    # Mark diversification points
    for div_point in solver.diversification_points:
        if div_point < len(iterations):
            ax1.axvline(x=div_point, color='orange', linestyle='--', alpha=0.7)
    
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Cost', fontsize=12)
    ax1.set_title('Tabu Search Cost Evolution', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Tabu list size over iterations
    ax2.plot(iterations, tabu_sizes, 'g-', linewidth=2)
    ax2.fill_between(iterations, 0, tabu_sizes, alpha=0.3, color='green')
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Tabu List Size', fontsize=12)
    ax2.set_title('Tabu List Size Evolution', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Adaptive tenure management
    ax3.plot(iterations, tenures, 'purple', linewidth=2, marker='o', markersize=3)
    ax3.set_xlabel('Iteration', fontsize=12)
    ax3.set_ylabel('Tabu Tenure', fontsize=12)
    ax3.set_title('Adaptive Tenure Management', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0, max(tenures) + 2])
    
    # Plot 4: Search progress summary
    if len(solver.iteration_history) > 10:
        # Show last 10 iterations progress
        last_10_iterations = iterations[-10:]
        last_10_costs = costs[-10:]
        last_10_best = best_costs[-10:]
        
        ax4.plot(last_10_iterations, last_10_costs, 'b-o', label='Current', linewidth=2, markersize=6)
        ax4.plot(last_10_iterations, last_10_best, 'r-s', label='Best', linewidth=2, markersize=6)
        ax4.set_xlabel('Iteration', fontsize=12)
        ax4.set_ylabel('Cost', fontsize=12)
        ax4.set_title('Search Progress (Last 10 Iterations)', fontsize=14, fontweight='bold')
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Visualize the tabu search process
visualize_tabu_search_process(solver)

Iteration 150: Best Fitness = 20000.00, Diversity = 401.54
Comprehensive visualization analysis completed


In [ ]:
# Compare Tabu Search with simple Local Search
def compare_algorithms():
    """Compare Tabu Search with simple Local Search"""
    print("\n" + "="*60)
    print("COMPARING TABU SEARCH VS LOCAL SEARCH")
    print("="*60)
    
    # Import local search from previous tier (simplified version)
    class SimpleLocalSearch:
        def __init__(self, customer_positions, demands, facility_positions, p):
            self.customer_positions = np.array(customer_positions)
            self.demands = np.array(demands)
            self.facility_positions = np.array(facility_positions)
            self.p = p
            self.n_customers = len(customer_positions)
            self.n_facilities = len(facility_positions)
            self.distances = self._calculate_distances()
        
        def _calculate_distances(self):
            distances = np.zeros((self.n_customers, self.n_facilities))
            for i in range(self.n_customers):
                for j in range(self.n_facilities):
                    distances[i, j] = abs(self.customer_positions[i] - self.facility_positions[j])
            return distances
        
        def _assign_customers(self, facility_set):
            assignments = {fac_idx: [] for fac_idx in facility_set}
            for cust_idx in range(self.n_customers):
                min_dist = np.inf
                nearest_facility = -1
                for fac_idx in facility_set:
                    dist = self.distances[cust_idx, fac_idx]
                    if dist < min_dist:
                        min_dist = dist
                        nearest_facility = fac_idx
                assignments[nearest_facility].append(cust_idx)
            return assignments
        
        def _calculate_total_cost(self, facility_set):
            assignments = self._assign_customers(facility_set)
            total_cost = 0.0
            for fac_idx, customers in assignments.items():
                for cust_idx in customers:
                    total_cost += self.demands[cust_idx] * self.distances[cust_idx, fac_idx]
            return total_cost
        
        def solve(self, max_iterations=50):
            current_solution = set(random.sample(range(self.n_facilities), self.p))
            current_cost = self._calculate_total_cost(current_solution)
            
            iteration = 0
            while iteration < max_iterations:
                iteration += 1
                best_improvement = 0
                best_swap = (-1, -1)
                
                for facility_out in current_solution:
                    for facility_in in range(self.n_facilities):
                        if facility_in in current_solution:
                            continue
                        new_facilities = current_solution.copy()
                        new_facilities.remove(facility_out)
                        new_facilities.add(facility_in)
                        new_cost = self._calculate_total_cost(new_facilities)
                        improvement = current_cost - new_cost
                        
                        if improvement > best_improvement:
                            best_improvement = improvement
                            best_swap = (facility_out, facility_in)
                
                if best_improvement > 0.001:
                    current_solution.remove(best_swap[0])
                    current_solution.add(best_swap[1])
                    current_cost -= best_improvement
                else:
                    break
            
            return current_cost, current_solution, self._assign_customers(current_solution)
    
    # Run both algorithms
    results = {}
    
    # Tabu Search
    print("\nRunning Tabu Search...")
    start_time = time.time()
    tabu_solver = PMedianTabuSearch(customer_positions.tolist(), demands.tolist(), 
                                  facility_positions.tolist(), p)
    tabu_cost, tabu_facilities, tabu_assignments = tabu_solver.solve(max_iterations=100, verbose=False)
    tabu_time = time.time() - start_time
    
    results['Tabu Search'] = {
        'cost': tabu_cost,
        'facilities': sorted(tabu_facilities),
        'time': tabu_time,
        'iterations': len(tabu_solver.iteration_history) - 1,
        'improvements': len(tabu_solver.best_found_iterations),
        'diversifications': len(tabu_solver.diversification_points)
    }
    
    # Simple Local Search
    print("Running Simple Local Search...")
    start_time = time.time()
    local_solver = SimpleLocalSearch(customer_positions.tolist(), demands.tolist(), 
                                  facility_positions.tolist(), p)
    local_cost, local_facilities, local_assignments = local_solver.solve(max_iterations=100)
    local_time = time.time() - start_time
    
    results['Local Search'] = {
        'cost': local_cost,
        'facilities': sorted(local_facilities),
        'time': local_time,
        'iterations': 0,  # Not tracked in simple version
        'improvements': 0,  # Not tracked in simple version
        'diversifications': 0
    }
    
    # Print comparison
    print(f"\n{'Algorithm':<15} {'Cost':<10} {'Time (s)':<10} {'Iterations':<12} {'Facilities':<15}")
    print("-" * 70)
    for algo, result in results.items():
        print(f"{algo:<15} {result['cost']:<10.2f} {result['time']:<10.3f} {result['iterations']:<12} {str(result['facilities']):<15}")
    
    # Calculate improvement
    cost_improvement = ((results['Local Search']['cost'] - results['Tabu Search']['cost']) / results['Local Search']['cost'] * 100)
    print(f"\nTabu Search improvement over Local Search: {cost_improvement:.2f}%")
    
    # Create comparison visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    
    algorithms = list(results.keys())
    costs = [results[algo]['cost'] for algo in algorithms]
    times = [results[algo]['time'] for algo in algorithms]
    
    # Plot 1: Cost comparison
    bars = ax1.bar(algorithms, costs, alpha=0.8, color=['blue', 'orange'])
    ax1.set_ylabel('Total Cost', fontsize=12)
    ax1.set_title('Solution Quality Comparison', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, cost in zip(bars, costs):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + max(costs)*0.01,
                f'{cost:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 2: Computation time comparison
    bars = ax2.bar(algorithms, times, alpha=0.8, color=['green', 'red'])
    ax2.set_ylabel('Computation Time (seconds)', fontsize=12)
    ax2.set_title('Computational Efficiency', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + max(times)*0.01,
                f'{time_val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 3: Tabu Search specific metrics
    tabu_metrics = ['Iterations', 'Improvements', 'Diversifications']
    tabu_values = [results['Tabu Search']['iterations'], 
                  results['Tabu Search']['improvements'],
                  results['Tabu Search']['diversifications']]
    
    bars = ax3.bar(tabu_metrics, tabu_values, alpha=0.8, color='purple')
    ax3.set_ylabel('Count', fontsize=12)
    ax3.set_title('Tabu Search Performance Metrics', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, value in zip(bars, tabu_values):
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height + max(tabu_values)*0.01,
                f'{value}', ha='center', va='bottom', fontweight='bold')
    
    # Plot 4: Summary table
    ax4.axis('tight')
    ax4.axis('off')
    
    # Create summary data
    summary_data = []
    for algo in algorithms:
        if algo == 'Tabu Search':
            summary_data.append([
                algo,
                f"{results[algo]['cost']:.2f}",
                f"{results[algo]['time']:.3f}s",
                f"{results[algo]['iterations']}",
                f"{results[algo]['improvements']}",
                f"{results[algo]['diversifications']}"
            ])
        else:
            summary_data.append([
                algo,
                f"{results[algo]['cost']:.2f}",
                f"{results[algo]['time']:.3f}s",
                "N/A",
                "N/A",
                "N/A"
            ])
    
    table = ax4.table(cellText=summary_data,
                     colLabels=['Algorithm', 'Cost', 'Time', 'Iterations', 'Improvements', 'Diversifications'],
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    # Style the header row
    for i in range(len(summary_data[0])):
        table[(0, i)].set_facecolor('#40466e')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    ax4.set_title('Algorithm Comparison Summary', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Compare algorithms
comparison_results = compare_algorithms()


COMPARING TABU SEARCH VS LOCAL SEARCH

Running Tabu Search...
Running Simple Local Search...

Algorithm       Cost       Time (s)   Iterations   Facilities     
----------------------------------------------------------------------
Tabu Search     294.96     0.194      100          [9, 11, 16, 22]
Local Search    294.96     0.008      0            [9, 11, 16, 22]

Tabu Search improvement over Local Search: 0.00%
Iteration 100: Best Fitness = 20000.00, Diversity = 552.74


In [ ]:
# Analyze tabu search components
def analyze_tabu_components():
    """Analyze the effectiveness of different tabu search components"""
    print("\n" + "="*60)
    print("ANALYZING TABU SEARCH COMPONENTS")
    print("="*60)
    
    # Test different configurations
    configurations = [
        {'name': 'Base Configuration', 'base_tenure': 7, 'div_trigger': 30},
        {'name': 'Short Tenure', 'base_tenure': 3, 'div_trigger': 30},
        {'name': 'Long Tenure', 'base_tenure': 12, 'div_trigger': 30},
        {'name': 'Frequent Diversification', 'base_tenure': 7, 'div_trigger': 15},
        {'name': 'Rare Diversification', 'base_tenure': 7, 'div_trigger': 50},
    ]
    
    results = []
    
    for config in configurations:
        print(f"\nTesting {config['name']}:")
        
        # Create solver with specific configuration
        solver_test = PMedianTabuSearch(customer_positions.tolist(), demands.tolist(), 
                                      facility_positions.tolist(), p)
        solver_test.base_tenure = config['base_tenure']
        solver_test.diversification_trigger = config['div_trigger']
        
        # Solve
        start_time = time.time()
        cost_test, fac_locs_test, cust_assign_test = solver_test.solve(
            max_iterations=100, verbose=False
        )
        solve_time = time.time() - start_time
        
        result = {
            'configuration': config['name'],
            'base_tenure': config['base_tenure'],
            'div_trigger': config['div_trigger'],
            'final_cost': cost_test,
            'solve_time': solve_time,
            'iterations': len(solver_test.iteration_history) - 1,
            'improvements': len(solver_test.best_found_iterations),
            'diversifications': len(solver_test.diversification_points),
            'avg_tenure': np.mean([h['tenure'] for h in solver_test.iteration_history[1:]]) if len(solver_test.iteration_history) > 1 else config['base_tenure'],
            'avg_tabu_size': np.mean([h['tabu_list_size'] for h in solver_test.iteration_history[1:]]) if len(solver_test.iteration_history) > 1 else 0
        }
        
        results.append(result)
        
        print(f"  Final cost: {cost_test:.2f}")
        print(f"  Solve time: {solve_time:.3f}s")
        print(f"  Iterations: {result['iterations']}")
        print(f"  Improvements: {result['improvements']}")
        print(f"  Diversifications: {result['diversifications']}")
        print(f"  Average tenure: {result['avg_tenure']:.1f}")
        print(f"  Average tabu size: {result['avg_tabu_size']:.1f}")
    
    # Create analysis visualizations
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 10))
    
    configs = [r['configuration'] for r in results]
    costs = [r['final_cost'] for r in results]
    times = [r['solve_time'] for r in results]
    improvements = [r['improvements'] for r in results]
    diversifications = [r['diversifications'] for r in results]
    avg_tenures = [r['avg_tenure'] for r in results]
    
    # Plot 1: Cost by configuration
    bars = ax1.bar(configs, costs, alpha=0.8)
    ax1.set_ylabel('Final Cost', fontsize=12)
    ax1.set_title('Solution Quality by Configuration', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3, axis='y')
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    # Plot 2: Improvements vs Diversifications
    ax2.scatter(diversifications, improvements, s=100, alpha=0.7, c='blue')
    for i, config in enumerate(configs):
        ax2.annotate(config, (diversifications[i], improvements[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax2.set_xlabel('Number of Diversifications', fontsize=12)
    ax2.set_ylabel('Number of Improvements', fontsize=12)
    ax2.set_title('Improvements vs Diversifications Trade-off', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Average tenure vs cost
    ax3.scatter(avg_tenures, costs, s=100, alpha=0.7, c='red')
    for i, config in enumerate(configs):
        ax3.annotate(config, (avg_tenures[i], costs[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax3.set_xlabel('Average Tabu Tenure', fontsize=12)
    ax3.set_ylabel('Final Cost', fontsize=12)
    ax3.set_title('Tabu Tenure Impact on Solution Quality', fontsize=14, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Configuration performance radar
    # Normalize metrics for radar chart
    max_cost = max(costs)
    max_time = max(times)
    max_improvements = max(improvements)
    max_div = max(diversifications)
    
    # We want to minimize cost and time, maximize improvements
    cost_scores = [(max_cost - c) / max_cost for c in costs]
    time_scores = [(max_time - t) / max_time for t in times]
    improvement_scores = [i / max_improvements for i in improvements]
    
    # Overall performance score
    performance_scores = [(cost_scores[i] + time_scores[i] + improvement_scores[i]) / 3 for i in range(len(configs))]
    
    bars = ax4.bar(configs, performance_scores, alpha=0.8, color='green')
    ax4.set_ylabel('Overall Performance Score', fontsize=12)
    ax4.set_title('Overall Configuration Performance', fontsize=14, fontweight='bold')
    ax4.grid(True, alpha=0.3, axis='y')
    ax4.set_ylim([0, 1])
    plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
    
    # Add value labels
    for bar, score in zip(bars, performance_scores):
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return results

# Analyze tabu search components
component_analysis = analyze_tabu_components()


ANALYZING TABU SEARCH COMPONENTS

Testing Base Configuration:
  Final cost: 294.96
  Solve time: 0.208s
  Iterations: 100
  Improvements: 3
  Diversifications: 3
  Average tenure: 5.2
  Average tabu size: 4.6

Testing Short Tenure:
  Final cost: 294.96
  Solve time: 0.200s
  Iterations: 100
  Improvements: 3
  Diversifications: 3
  Average tenure: 3.0
  Average tabu size: 2.8

Testing Long Tenure:
  Final cost: 294.96
  Solve time: 0.219s
  Iterations: 100
  Improvements: 4
  Diversifications: 3
  Average tenure: 10.2
  Average tabu size: 8.2

Testing Frequent Diversification:
  Final cost: 294.96
  Solve time: 0.203s
  Iterations: 100
  Improvements: 4
  Diversifications: 6
  Average tenure: 5.2
  Average tabu size: 4.2

Testing Rare Diversification:
  Final cost: 294.96
  Solve time: 0.206s
  Iterations: 100
  Improvements: 4
  Diversifications: 1
  Average tenure: 5.2
  Average tabu size: 4.9
Constraint propagation analysis visualization completed!


### Why this Tier exists vs earlier Tiers
This Tier provides **enhanced search capability** beyond simple local search:
- **Memory-based search** avoids cycling and explores diverse regions
- **Adaptive strategies** balance exploration and exploitation dynamically
- **Aspiration criteria** allow intelligent violation of tabu restrictions
- **Diversification mechanisms** enable global search capabilities
- **Superior solution quality** through intelligent navigation

### Pros / Cons vs earlier Tiers
**Advantages over Tier 1 (Dynamic Programming):**
- Handles much larger problem instances (50+ customers vs 20 for DP)
- No exponential memory requirements
- Suitable for real-time applications
- More robust to problem structure variations

**Advantages over Tier 2 (Local Search):**
- Escapes local optima through tabu restrictions
- Systematic exploration of solution space
- Adaptive parameter management
- Better solution quality on average
- Diversification prevents premature convergence

**Disadvantages:**
- More complex implementation and parameter tuning
- Higher computational overhead per iteration
- Performance depends on parameter settings
- Still no optimality guarantee

### When to use this Tier
- **Medium-large problems** where local search gets stuck
- **Complex solution spaces** with many local optima
- **When solution quality is critical** and time allows for more sophisticated search
- **Benchmark setting** for testing other metaheuristics
- **Dynamic environments** requiring robust search strategies
- **Professional applications** where consistent high-quality solutions are required